In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
from torchvision import datasets

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using {device} device")

Using cpu device


In [2]:
train_raw = datasets.MNIST(root="data", train=True,  download=True)
test_raw  = datasets.MNIST(root="data", train=False, download=True)

def prep(raw):
    X = raw.data.float() / 255.0      # (N, 28, 28), pixel values in [0, 1]
    X = (X - 0.1307) / 0.3081         # normalize
    X = X.view(len(X), -1)            # flatten each image to a 784-length vector
    y = raw.targets.clone()           # integer labels 0–9
    return X, y

X_train, y_train = prep(train_raw)
X_test,  y_test  = prep(test_raw)

print(X_train.shape, y_train.shape)
print(X_test.shape,  y_test.shape)

100%|██████████| 9.91M/9.91M [00:01<00:00, 6.05MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 160kB/s]
100%|██████████| 1.65M/1.65M [00:01<00:00, 1.52MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 6.40MB/s]


torch.Size([60000, 784]) torch.Size([60000])
torch.Size([10000, 784]) torch.Size([10000])


In [3]:
class MLP(nn.Module):
    def __init__(self, hidden=(1200, 1200), dropout=0.0, input_dropout=0.0):
        super().__init__()
        layers = [nn.Dropout(input_dropout)]
        in_dim = 784
        for h in hidden:
            layers += [nn.Linear(in_dim, h), nn.ReLU(), nn.Dropout(dropout)]
            in_dim = h
        layers.append(nn.Linear(in_dim, 10))   # final layer: 10 logits, NO softmax
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)   # returns raw logits z

teacher_test = MLP(hidden=(1200, 1200))
print(teacher_test)
print("Parameters:", sum(p.numel() for p in teacher_test.parameters()))

MLP(
  (net): Sequential(
    (0): Dropout(p=0.0, inplace=False)
    (1): Linear(in_features=784, out_features=1200, bias=True)
    (2): ReLU()
    (3): Dropout(p=0.0, inplace=False)
    (4): Linear(in_features=1200, out_features=1200, bias=True)
    (5): ReLU()
    (6): Dropout(p=0.0, inplace=False)
    (7): Linear(in_features=1200, out_features=10, bias=True)
  )
)
Parameters: 2395210


In [4]:
@torch.no_grad()
def evaluate(model, X, y, batch_size=1000):
    """Returns the NUMBER of misclassified examples (the paper reports errors, not accuracy)."""
    model.eval()                                  # turn dropout OFF
    errors = 0
    for i in range(0, len(X), batch_size):
        xb = X[i:i+batch_size].to(device)
        yb = y[i:i+batch_size].to(device)
        preds = model(xb).argmax(dim=1)           # argmax of logits = predicted class
        errors += (preds != yb).sum().item()
    return errors


def train(model, loader, compute_loss, epochs=20, lr=1e-3, verbose=True):
    model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    for epoch in range(epochs):
        model.train()                             # turn dropout ON
        total = 0.0
        for batch in loader:
            batch = [t.to(device) for t in batch]
            loss = compute_loss(model, batch)
            opt.zero_grad()                       # 1. clear old gradients
            loss.backward()                       # 2. compute new gradients
            opt.step()                            # 3. update weights
            total += loss.item()
        if verbose and ((epoch + 1) % 5 == 0 or epoch == 0):
            errs = evaluate(model, X_test, y_test)
            print(f"epoch {epoch+1:3d} | avg loss {total/len(loader):.4f} | test errors {errs}")
    return model


def hard_loss(model, batch):
    x, y = batch[0], batch[1]
    return F.cross_entropy(model(x), y)           # standard T = 1 cross-entropy

In [5]:
hard_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=128, shuffle=True)

torch.manual_seed(0)
teacher = MLP(hidden=(1200, 1200), dropout=0.5, input_dropout=0.2)
train(teacher, hard_loader, hard_loss, epochs=30)

print("TEACHER test errors:", evaluate(teacher, X_test, y_test))

epoch   1 | avg loss 0.3073 | test errors 370
epoch   5 | avg loss 0.1266 | test errors 215
epoch  10 | avg loss 0.1041 | test errors 207
epoch  15 | avg loss 0.0970 | test errors 175
epoch  20 | avg loss 0.0895 | test errors 151
epoch  25 | avg loss 0.0896 | test errors 172
epoch  30 | avg loss 0.0870 | test errors 171
TEACHER test errors: 171


In [6]:
torch.manual_seed(0)
baseline = MLP(hidden=(800, 800))          # smaller, and no dropout
train(baseline, hard_loader, hard_loss, epochs=20)

print("BASELINE STUDENT test errors:", evaluate(baseline, X_test, y_test))

epoch   1 | avg loss 0.2060 | test errors 351
epoch   5 | avg loss 0.0372 | test errors 201
epoch  10 | avg loss 0.0241 | test errors 190
epoch  15 | avg loss 0.0165 | test errors 180
epoch  20 | avg loss 0.0111 | test errors 167
BASELINE STUDENT test errors: 167


This is very surprising. The student network with 400 lesser neurons in each of the hidden layers outperformed the teacher network with dropout regularization in lesser epochs. I want to see what might happen if I run the reacher network again without dropout. Did dropout cause underfitting?

In [7]:
hard_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=128, shuffle=True)

torch.manual_seed(0)
nd_teacher = MLP(hidden=(1200, 1200))
train(nd_teacher, hard_loader, hard_loss, epochs=30)

print("TEACHER (without dropout) test errors:", evaluate(nd_teacher, X_test, y_test))

epoch   1 | avg loss 0.1958 | test errors 331
epoch   5 | avg loss 0.0372 | test errors 225
epoch  10 | avg loss 0.0177 | test errors 198
epoch  15 | avg loss 0.0150 | test errors 205
epoch  20 | avg loss 0.0110 | test errors 178
epoch  25 | avg loss 0.0089 | test errors 188
epoch  30 | avg loss 0.0090 | test errors 160
TEACHER (without dropout) test errors: 160


Okay, given the size of the dataset, the difference doesn't seem statistically significiant. Or is it? let us examine this further:

In the teacher network with dropout regularization between the epochs 20 and 25, the error jumps from 151 to 172 i.e, there is an oscillation. This could be because of sampling noise, or convergence, or due to less training time.
In the original paper, the authors ran thousands of epochs with better intitalization and a decaying learning rate. The results from 30 epochs with a constant learning rate is not enough to make conclusions on the nature of the neural net.

Let us dig into this deeper by measuring training error to see if underfitting is actually the culprit here.  

In [8]:
print("Dropout teacher    train errors:", evaluate(teacher,    X_train, y_train))
print("No-dropout teacher train errors:", evaluate(nd_teacher, X_train, y_train))
print("Baseline student   train errors:", evaluate(baseline,   X_train, y_train))

Dropout teacher    train errors: 247
No-dropout teacher train errors: 92
Baseline student   train errors: 161


Why is no dropout performing better than dropout and how is student doing so good?! This very puzzling - It is because training is being done a larger dataset, precisely 6 times as big, this makes sense. Since this is the number of errors, and not the error rate, this makes sense.

For the purposes of this experiment, I need to build a much stronger teacher model than the student. Like, in the paper, i am going to introduce some jitter to the input images, more epochs, and a decaying learning rate.

In [9]:
#Let us try and speed up the training first
class FastLoader:
    """Drop-in replacement for DataLoader when the whole dataset fits on the GPU."""
    def __init__(self, *tensors, batch_size=128):
        self.tensors = [t.to(device) for t in tensors]
        self.bs = batch_size
    def __len__(self):
        return (len(self.tensors[0]) + self.bs - 1) // self.bs
    def __iter__(self):
        idx = torch.randperm(len(self.tensors[0]), device=device)   # shuffle each epoch
        for i in range(0, len(idx), self.bs):
            j = idx[i:i+self.bs]
            yield [t[j] for t in self.tensors]                       # same rows from every tensor

hard_loader = FastLoader(X_train, y_train)

#### Combining Adam and Cosine anealing

Adam uses momentum from past gradients and adapts each parameter’s learning rate based on the recent magnitude of its gradients. Cosine annealing smoothly schedules the overall learning rate during training. Apparently, this is a powerful combination, and I am just learning about it.

In [10]:
# Defining training again to use decaying learning rate using cosine annealing and Adam

def train(model, loader, compute_loss, epochs=20, lr=1e-3, verbose=True, cosine=False):
    model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs) if cosine else None
    for epoch in range(epochs):
        model.train()
        total = 0.0
        for batch in loader:
            batch = [t.to(device) for t in batch]
            loss = compute_loss(model, batch)
            opt.zero_grad(); loss.backward(); opt.step()
            total += loss.item()
        if sched: sched.step()            # shrink the LR a bit after each epoch
        if verbose and ((epoch + 1) % 5 == 0 or epoch == 0):
            print(f"epoch {epoch+1:3d} | lr {opt.param_groups[0]['lr']:.2e} | "
                  f"avg loss {total/len(loader):.4f} | test errors {evaluate(model, X_test, y_test)}")
    return model

#### Add Jitter

Jitter is data augumentation. By shifting, rotating, and palying around with the data little, each image is slightly different which encourages the model to learn the patterns in the data rather than just memorizing each image.

In [11]:
# Add jitter

def jitter(x, max_shift=2):
    """Shift each image randomly by up to max_shift pixels (the paper's 'jitter')."""
    n = x.shape[0]
    bg = -0.1307 / 0.3081                          # a black pixel, after normalization
    imgs = (x - bg).view(n, 1, 28, 28)             # make background 0, so padding is black
    theta = torch.zeros(n, 2, 3, device=x.device)  # one 2x3 affine matrix per image
    theta[:, 0, 0] = 1
    theta[:, 1, 1] = 1                             # identity: no rotation or scaling
    theta[:, :, 2] = (torch.rand(n, 2, device=x.device) * 2 - 1) * max_shift * (2 / 28)  # translation
    grid = F.affine_grid(theta, imgs.shape, align_corners=False)
    out = F.grid_sample(imgs, grid, align_corners=False)
    return out.view(n, 784) + bg

def hard_loss_jitter(model, batch):
    x, y = batch[0], batch[1]
    return F.cross_entropy(model(jitter(x)), y)    # a freshly shifted copy every time

#### Training
Now train the teacher on 60 epochs

In [12]:
torch.manual_seed(0)
teacher2 = MLP(hidden=(1200, 1200), dropout=0.5, input_dropout=0.2)
train(teacher2, hard_loader, hard_loss_jitter, epochs=60, cosine=True)
print("TEACHER v2 test errors:", evaluate(teacher2, X_test, y_test))

epoch   1 | lr 9.99e-04 | avg loss 0.4601 | test errors 331
epoch   5 | lr 9.83e-04 | avg loss 0.1934 | test errors 220
epoch  10 | lr 9.33e-04 | avg loss 0.1752 | test errors 165
epoch  15 | lr 8.54e-04 | avg loss 0.1554 | test errors 159
epoch  20 | lr 7.50e-04 | avg loss 0.1339 | test errors 130
epoch  25 | lr 6.29e-04 | avg loss 0.1270 | test errors 126
epoch  30 | lr 5.00e-04 | avg loss 0.1093 | test errors 122
epoch  35 | lr 3.71e-04 | avg loss 0.0981 | test errors 106
epoch  40 | lr 2.50e-04 | avg loss 0.0857 | test errors 95
epoch  45 | lr 1.46e-04 | avg loss 0.0770 | test errors 94
epoch  50 | lr 6.70e-05 | avg loss 0.0686 | test errors 87
epoch  55 | lr 1.70e-05 | avg loss 0.0681 | test errors 81
epoch  60 | lr 0.00e+00 | avg loss 0.0631 | test errors 81
TEACHER v2 test errors: 81


Okay, this is quite good. The test errors went down significantly. The paper reports 67 errors, but tehy did train on even more eopchs. This will hopefully do for now. We can retrain again if the difference between the student network and this one is not significant enough.

In [13]:
torch.manual_seed(0)
baseline2 = MLP(hidden=(800, 800))
train(baseline2, hard_loader, hard_loss, epochs=20, cosine=True)
print("BASELINE v2 test errors:", evaluate(baseline2, X_test, y_test))

epoch   1 | lr 9.94e-04 | avg loss 0.2022 | test errors 350
epoch   5 | lr 8.54e-04 | avg loss 0.0311 | test errors 217
epoch  10 | lr 5.00e-04 | avg loss 0.0084 | test errors 184
epoch  15 | lr 1.46e-04 | avg loss 0.0001 | test errors 130
epoch  20 | lr 0.00e+00 | avg loss 0.0000 | test errors 128
BASELINE v2 test errors: 128


I wonder why the baseline at 20 epochs has less errors than the teacher at the same epoch. let us explore this further.

At epoch 20 the baseline (128) looks comparable to the teacher (130), but that’s likely not not a fair comparison because the cosine schedule had finished for the 20-epoch baseline, while the 60-epoch teacher was still at a high learning rate. The teacher also trains on a harder, noisier task (dropout + jitter), which slows early progress but improves final generalization. Comparing final models (81 vs 128 errors) shows the teacher is clearly better, well beyond sampling noise.

For now, let us move on to distillation to see if the distilled network performs better than the baseline student.

In [15]:
# define some helper function for distillation

@torch.no_grad()
def get_logits(model, X, batch_size=1000):
    """Run a trained model over X and return its raw logits (no softmax)."""
    model.eval()                                    # dropout OFF: the teacher's real opinion
    out = []
    for i in range(0, len(X), batch_size):
        out.append(model(X[i:i+batch_size].to(device)).cpu())
    return torch.cat(out)


torch.set_printoptions(precision=4, sci_mode=False)

def show_soft_targets(i):
    """Print the teacher's softened distribution for training image i at several temperatures."""
    print(f"Image {i}, true label = {y_train[i].item()}")
    for T in [1, 2, 5, 20]:
        probs = F.softmax(teacher_logits[i] / T, dim=0)
        entropy = -(probs * probs.clamp_min(1e-12).log()).sum().item()
        print(f"  T={T:>2}  entropy={entropy:.3f}  {probs}")


def distillation_loss(student_logits, teacher_logits, labels, T, alpha):
    # Soft loss: cross-entropy between teacher and student, both softened by the same T
    soft_targets      = F.softmax(teacher_logits / T, dim=1)
    student_log_probs = F.log_softmax(student_logits / T, dim=1)
    soft_loss = -(soft_targets * student_log_probs).sum(dim=1).mean()

    # Hard loss: ordinary cross-entropy with the true labels, at T = 1
    hard_loss = F.cross_entropy(student_logits, labels)

    # T² keeps the soft gradients from shrinking as T grows
    return alpha * (T ** 2) * soft_loss + (1 - alpha) * hard_loss


def make_distill_loss(T, alpha):
    """Builds a loss function with T and alpha baked in, in the form train() expects."""
    def compute(model, batch):
        x, y, t_logits = batch
        return distillation_loss(model(x), t_logits, y, T, alpha)
    return compute

In [16]:
teacher_logits = get_logits(teacher2, X_train)     # from the NEW teacher (81 errors)
print(teacher_logits.shape)                        # torch.Size([60000, 10])

show_soft_targets(0)                               # a typical image

most_ambiguous = F.softmax(teacher_logits, dim=1).max(dim=1).values.argmin().item()
show_soft_targets(most_ambiguous)                  # the image the teacher is least sure about

torch.Size([60000, 10])
Image 0, true label = 5
  T= 1  entropy=0.010  tensor([0.0000, 0.0000, 0.0000, 0.0014, 0.0000, 0.9986, 0.0000, 0.0000, 0.0000,
        0.0000])
  T= 2  entropy=0.157  tensor([0.0000, 0.0000, 0.0000, 0.0358, 0.0000, 0.9639, 0.0000, 0.0000, 0.0001,
        0.0001])
  T= 5  entropy=0.754  tensor([0.0002, 0.0028, 0.0022, 0.2004, 0.0003, 0.7482, 0.0013, 0.0006, 0.0221,
        0.0221])
  T=20  entropy=2.065  tensor([0.0330, 0.0673, 0.0634, 0.1965, 0.0387, 0.2731, 0.0553, 0.0464, 0.1132,
        0.1132])
Image 48975, true label = 2
  T= 1  entropy=2.095  tensor([0.0181, 0.2212, 0.2143, 0.0672, 0.0925, 0.0666, 0.0486, 0.1087, 0.1213,
        0.0416])
  T= 2  entropy=2.243  tensor([0.0449, 0.1572, 0.1548, 0.0866, 0.1016, 0.0863, 0.0737, 0.1102, 0.1164,
        0.0682])
  T= 5  entropy=2.292  tensor([0.0737, 0.1216, 0.1208, 0.0958, 0.1021, 0.0957, 0.0898, 0.1055, 0.1078,
        0.0871])
  T=20  entropy=2.302  tensor([0.0928, 0.1052, 0.1050, 0.0991, 0.1007, 0.0991, 0.097

This is exactly like the paper described and it is very interesting to see it here.

For the true label 5, the teacher is pretty certain of its prediction 5 (0.9986) at temperature 1. A slight chance that it it might be three, but not enough for cross entropy to feel it. At temperatyre 5, all of the numbers have positive probabilities although they are not on a comaparable scale. At temperature 20, 5 still persists as the leading probabilities as it must, but the probabilities are on a much more comparable scale. These values encode infromation regarding similarity patters that the student model can learn.

We can observe similar results for the true label 2 as well. Unlike 5, all the classes have a positive probabilities at temperature 1, but scaling the temperature higher brigns them to a comparable scale. Nice!!

Now, let us move on to distillation.



In [18]:
distill_loader = FastLoader(X_train, y_train, teacher_logits)   # each row: image, label, teacher logits

torch.manual_seed(0)
student2 = MLP(hidden=(800, 800))                  # same architecture as baseline2
train(student2, distill_loader, make_distill_loss(T=20, alpha=0.9), epochs=20, cosine=True)

epoch   1 | lr 9.94e-04 | avg loss 777.9087 | test errors 239
epoch   5 | lr 8.54e-04 | avg loss 773.4823 | test errors 136
epoch  10 | lr 5.00e-04 | avg loss 773.2330 | test errors 118
epoch  15 | lr 1.46e-04 | avg loss 773.1273 | test errors 109
epoch  20 | lr 0.00e+00 | avg loss 773.0908 | test errors 111


MLP(
  (net): Sequential(
    (0): Dropout(p=0.0, inplace=False)
    (1): Linear(in_features=784, out_features=800, bias=True)
    (2): ReLU()
    (3): Dropout(p=0.0, inplace=False)
    (4): Linear(in_features=800, out_features=800, bias=True)
    (5): ReLU()
    (6): Dropout(p=0.0, inplace=False)
    (7): Linear(in_features=800, out_features=10, bias=True)
  )
)

In [19]:
print("=== RESULTS (test errors out of 10,000) ===")
print(f"Teacher v2        (1200x1200, dropout + jitter): {evaluate(teacher2,  X_test, y_test)}")
print(f"Baseline student  (800x800, hard labels)       : {evaluate(baseline2, X_test, y_test)}")
print(f"Distilled student (800x800, T=20, alpha=0.9)   : {evaluate(student2,  X_test, y_test)}")

=== RESULTS (test errors out of 10,000) ===
Teacher v2        (1200x1200, dropout + jitter): 81
Baseline student  (800x800, hard labels)       : 128
Distilled student (800x800, T=20, alpha=0.9)   : 111


Distlled student clearly performes better than baseline student, but the difference doesn't seem like a lot. why? I was expecting even better results.

Let us explore some more by jittering the data in the transfer set.

In [20]:
def make_online_distill_loss(teacher, T, alpha, max_shift=2):
    teacher.eval()                                  # teacher stays frozen, dropout OFF
    def compute(model, batch):
        x, y = batch[0], batch[1]
        xj = jitter(x, max_shift)                   # ONE shift, shared by teacher and student
        with torch.no_grad():                       # no gradients for the teacher
            t_logits = teacher(xj)
        return distillation_loss(model(xj), t_logits, y, T, alpha)
    return compute

In [21]:
torch.manual_seed(0)
baseline_j = MLP(hidden=(800, 800))
train(baseline_j, hard_loader, hard_loss_jitter, epochs=40, cosine=True, verbose=True);
print("BASELINE + jitter test errors:", evaluate(baseline_j, X_test, y_test))

epoch   1 | lr 9.98e-04 | avg loss 0.2920 | test errors 448
epoch   5 | lr 9.62e-04 | avg loss 0.0764 | test errors 185
epoch  10 | lr 8.54e-04 | avg loss 0.0523 | test errors 165
epoch  15 | lr 6.91e-04 | avg loss 0.0351 | test errors 133
epoch  20 | lr 5.00e-04 | avg loss 0.0244 | test errors 124
epoch  25 | lr 3.09e-04 | avg loss 0.0150 | test errors 117
epoch  30 | lr 1.46e-04 | avg loss 0.0079 | test errors 90
epoch  35 | lr 3.81e-05 | avg loss 0.0050 | test errors 86
epoch  40 | lr 0.00e+00 | avg loss 0.0040 | test errors 76
BASELINE + jitter test errors: 76


In [22]:
torch.manual_seed(0)
student_j = MLP(hidden=(800, 800))
train(student_j, hard_loader, make_online_distill_loss(teacher2, T=20, alpha=0.9),
      epochs=40, cosine=True);
print("DISTILLED + jitter test errors:", evaluate(student_j, X_test, y_test))

epoch   1 | lr 9.98e-04 | avg loss 785.7175 | test errors 232
epoch   5 | lr 9.62e-04 | avg loss 780.6440 | test errors 121
epoch  10 | lr 8.54e-04 | avg loss 780.2728 | test errors 122
epoch  15 | lr 6.91e-04 | avg loss 780.1318 | test errors 109
epoch  20 | lr 5.00e-04 | avg loss 780.2596 | test errors 97
epoch  25 | lr 3.09e-04 | avg loss 780.2113 | test errors 99
epoch  30 | lr 1.46e-04 | avg loss 780.0901 | test errors 99
epoch  35 | lr 3.81e-05 | avg loss 780.1314 | test errors 94
epoch  40 | lr 0.00e+00 | avg loss 780.1906 | test errors 96
DISTILLED + jitter test errors: 96


In [23]:
print("=== RESULTS (test errors out of 10,000) ===")
print(f"Teacher v2                      : {evaluate(teacher2,   X_test, y_test)}")
print(f"Baseline (no jitter)            : {evaluate(baseline2,  X_test, y_test)}")
print(f"Distilled (no jitter)           : {evaluate(student2,   X_test, y_test)}")
print(f"Baseline + jitter               : {evaluate(baseline_j, X_test, y_test)}")
print(f"Distilled + jitter (consistent) : {evaluate(student_j,  X_test, y_test)}")

=== RESULTS (test errors out of 10,000) ===
Teacher v2                      : 81
Baseline (no jitter)            : 128
Distilled (no jitter)           : 111
Baseline + jitter               : 76
Distilled + jitter (consistent) : 96


This is getting weirder. When jitter is involved, the baseline performs better than the distilled model? what! why?!